# Titanic Survival Prediction
Example of using Hyperopt to automatically find the best hyperparameters.


In [0]:
# Hyperopt isn't pre-installed on serverless compute.
%pip install hyperopt

In [0]:
import mlflow
import numpy as np
import pandas as pd

from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [0]:
# Load the data

data_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(data_url)

# Preprocessing
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

# Select core features and target
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
X = df[features]
y = df["Survived"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Define the hyperparameter search space

search_space = {
    "n_estimators": hp.quniform("n_estimators", 50, 300, 10),
    "max_depth": hp.quniform("max_depth", 3, 15, 1),
    "min_samples_split": hp.quniform("min_samples_split", 2, 10, 1),
    "criterion": hp.choice("criterion", ["gini", "entropy"]),
}

In [0]:
# Define the objective function

def objective(params):
    # Hyperopt returns quniform variables as floats; cast them to ints
    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"] = int(params["max_depth"])
    params["min_samples_split"] = int(params["min_samples_split"])

    # Enable MLflow nested autologging to capture details for every individual trial
    mlflow.sklearn.autolog()

    with mlflow.start_run(nested=True):
        # Initialize and fit
        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_train, y_train)

        # Predict and evaluate
        predictions = model.predict(X_test)
        prob_predictions = model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, predictions)
        auc = roc_auc_score(y_test, prob_predictions)
        # Return negative AUC as the loss to minimize
        return {"loss": -auc, "status": STATUS_OK, "accuracy": accuracy}

In [0]:
# Run Hyperopt evaluations locally (SparkTrials requires SparkContext, unavailable on serverless)
trials = Trials()

best_hyperparams = None

with mlflow.start_run(run_name="Titanic_Hyperopt_RandomForest") as parent_run:
    best_hyperparams = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=15,  # Tries 15 different parameter combinations
        trials=trials,
    )

    print("--- Tuning Complete ---")
    print("Best hyperparameters found index/values:", best_hyperparams)

In [0]:
# Map hyperopt indices to actual parameter values
criterion_options = ["gini", "entropy"]
best_params = {
    "n_estimators": int(best_hyperparams["n_estimators"]),
    "max_depth": int(best_hyperparams["max_depth"]),
    "min_samples_split": int(best_hyperparams["min_samples_split"]),
    "criterion": criterion_options[best_hyperparams["criterion"]],
}

with mlflow.start_run(run_name="titanic_best_model"):

    # Train a final model with the best hyperparameters
    best_model = RandomForestClassifier(**best_params, random_state=42)
    best_model.fit(X_train, y_train)

    # Evaluate the final model
    predictions = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Final model accuracy: {accuracy:.4f}")

    # Log and register the final model
    signature = infer_signature(X_test, best_model.predict(X_test))
    mlflow.sklearn.log_model(
        best_model, "model", signature=signature, registered_model_name="titanic_best_model"
    )